In [ ]:
import numpy as np

from func import *
from models import *

In [ ]:
train_df = pd.read_csv("processed7/train_data.csv")
test_df = pd.read_csv("processed7/test_data.csv")

train_df["logClosePrice"] = np.log1p(train_df["ClosePrice"])
train_df.drop(columns=["ClosePrice"], inplace=True)
test_df["logClosePrice"] = np.log1p(test_df["ClosePrice"])
test_df.drop(columns=["ClosePrice"], inplace=True)

In [63]:
X_train = train_df.drop(columns=["logClosePrice"], axis=1)
y_train = train_df["logClosePrice"]
X_test = test_df.drop(columns=["logClosePrice"], axis=1)
y_test = test_df["logClosePrice"]

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

nunique = X_train[cat_cols].nunique(dropna=False)

low_card_cols = nunique[(nunique <= 20)].index.tolist()
high_card_cols = nunique[(nunique >= 20)].index.tolist()

## router

In [ ]:
en_full = make_model_pipeline(
    model=ElasticNet(alpha=0.01778279410038923,
                     l1_ratio=0.05),
    num_cols=num_cols,
    low_card_cols=low_card_cols,
    high_card_cols=high_card_cols
)

xgb_num = make_model_numeric_only_pipeline(
    model=XGBRegressor(n_estimators=1219,
                       learning_rate=0.05383345943137039,
                       max_depth=6,
                       subsample=0.755325405158244,
                       colsample_bytree=0.7844598129752072,
                       min_child_weight=14,
                       reg_lambda=4.736558858366902,
                       reg_alpha=1.10973369349242),
    num_cols=num_cols,
    num_scaler=None
)

# ---------- Stack them ----------
stack = StackingRegressor(
    estimators=[
        ("en_full", en_full),
        ("xgb_num", xgb_num),
    ],
    final_estimator=XGBRegressor(
        max_depth=2,
        n_estimators=200,
        learning_rate=0.05
    ),
    cv=5,
    n_jobs=-1
)

# Fit on log target
y_train = train_df["logClosePrice"]
stack.fit(X_train, y_train)

# Predict log target
X_test = test_df.drop("logClosePrice", axis=1)
y_test = test_df["logClosePrice"]
y_train_pred = stack.predict(X_train)
y_pred = stack.predict(X_test)

In [107]:
y_train_exp = np.expm1(y_train)
y_test_exp = np.expm1(y_test)
y_pred_exp = np.expm1(y_pred)

In [108]:
q1 = 0.95

train_cut = np.quantile(y_train_exp, q1)
test_cut = np.quantile(y_test_exp, q1)
pred_cut = np.quantile(y_pred_exp, q1)

In [104]:
y_test_assigment = (y_test_exp > test_cut).astype(int)
y_test_pred_assigment = (y_pred_exp > pred_cut).astype(int)

In [105]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_assigment, y_test_pred_assigment)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[9030  101]
 [ 100  377]]


In [106]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test_assigment,
    y_test_pred_assigment,
    target_names=["normal", "high"],
    digits=4
))

              precision    recall  f1-score   support

      normal     0.9890    0.9889    0.9890      9131
        high     0.7887    0.7904    0.7895       477

    accuracy                         0.9791      9608
   macro avg     0.8889    0.8896    0.8893      9608
weighted avg     0.9791    0.9791    0.9791      9608



# segment model

In [109]:
# Split data
X_train_normal = X_train[y_train_exp <= train_cut]
y_train_normal = y_train[y_train_exp <= train_cut]

X_train_high = X_train[y_train_exp > train_cut]
y_train_high = y_train[y_train_exp > train_cut]

In [127]:
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

# ---------- normal subset ----------
enet_normal_search = RandomizedSearchCV(
    make_model_pipeline(
        model=ElasticNet(max_iter=20000),
        num_cols=num_cols,
        low_card_cols=low_card_cols,
        high_card_cols=high_card_cols),
    param_distributions={
        "model__alpha": [0.0005, 0.001, 0.005, 0.01, 0.05],
        "model__l1_ratio": [0.05, 0.1, 0.3, 0.5, 0.8]
    },
    n_iter=20,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=42
)
enet_normal_search.fit(X_train_normal, y_train_normal)
best_enet_normal = enet_normal_search.best_estimator_

xgb_normal_search = RandomizedSearchCV(
    make_model_numeric_only_pipeline(
        model=XGBRegressor(random_state=42),
        num_cols=num_cols,
        num_scaler=None
    ),
    param_distributions={
        "model__n_estimators": [800, 1200, 1600],
        "model__max_depth": [5, 6, 7, 8],
        "model__learning_rate": [0.03, 0.05],
        "model__min_child_weight": [3, 5, 8],
        "model__subsample": [0.75, 0.85, 0.95],
        "model__colsample_bytree": [0.75, 0.85, 0.95],
        "model__reg_alpha": [0, 0.1, 1],
        "model__reg_lambda": [1, 3, 5]
    },
    n_iter=50,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=42
)
xgb_normal_search.fit(X_train_normal, y_train_normal)
best_xgb_normal = xgb_normal_search.best_estimator_

stack_normal = StackingRegressor(
    estimators=[
        ("enet", best_enet_normal),
        ("xgb", best_xgb_normal)
    ],
    final_estimator=XGBRegressor(
        max_depth=2,
        n_estimators=200,
        learning_rate=0.05
    ),
    cv=5,
    n_jobs=-1
)

stack_normal.fit(X_train_normal, y_train_normal)

,estimators,"[('enet', ...), ('xgb', ...)]"
,final_estimator,"XGBRegressor(...ree=None, ...)"
,cv,5
,n_jobs,-1
,passthrough,False
,verbose,0
,transformers,"[('num', ...), ('low', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None


In [128]:
# ---------- high subset ----------
enet_high_search = RandomizedSearchCV(
    make_model_pipeline(
        model=ElasticNet(max_iter=20000),
        num_cols=num_cols,
        low_card_cols=low_card_cols,
        high_card_cols=high_card_cols),
    param_distributions={
        "model__alpha": [0.0005, 0.001, 0.005, 0.01, 0.05],
        "model__l1_ratio": [0.05, 0.1, 0.3, 0.5, 0.8]
    },
    n_iter=20,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=42
)
enet_high_search.fit(X_train_high, y_train_high)
best_enet_high = enet_high_search.best_estimator_

xgb_high_search = RandomizedSearchCV(
    make_model_numeric_only_pipeline(
        model=XGBRegressor(random_state=42),
        num_cols=num_cols,
        num_scaler=None
    ),
    param_distributions={
        "model__n_estimators": [600, 1000, 1400],
        "model__max_depth": [4, 5, 6],
        "model__learning_rate": [0.02, 0.03, 0.05],
        "model__min_child_weight": [5, 8, 12],
        "model__subsample": [0.7, 0.8, 0.9],
        "model__colsample_bytree": [0.7, 0.8, 0.9],
        "model__reg_alpha": [0.1, 1, 3],
        "model__reg_lambda": [3, 5, 10]
    },
    n_iter=50,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=42
)
xgb_high_search.fit(X_train_high, y_train_high)
best_xgb_high = xgb_high_search.best_estimator_

stack_high = StackingRegressor(
    estimators=[
        ("enet", best_enet_high),
        ("xgb", best_xgb_high)
    ],
    final_estimator=XGBRegressor(
        max_depth=2,
        n_estimators=200,
        learning_rate=0.05
    ),
    cv=5,
    n_jobs=-1
)

stack_high.fit(X_train_high, y_train_high)

,estimators,"[('enet', ...), ('xgb', ...)]"
,final_estimator,"XGBRegressor(...ree=None, ...)"
,cv,5
,n_jobs,-1
,passthrough,False
,verbose,0
,transformers,"[('num', ...), ('low', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None


In [129]:
train_pred_normal = stack_normal.predict(X_train)
train_pred_high = stack_high.predict(X_train)

pred_normal = stack_normal.predict(X_test)
pred_high = stack_high.predict(X_test)

In [130]:
y_train_pred = stack.predict(X_train)
y_train_pred_exp = np.expm1(y_train_pred)
train_pred_cut = np.quantile(y_train_pred_exp, q1)
is_high_train = y_train_pred_exp > train_pred_cut

final_train_pred_log = np.where(
    is_high_train,
    train_pred_high,
    train_pred_normal
)

In [131]:
y_pred = stack.predict(X_test)
y_pred_exp = np.expm1(y_pred)
pred_cut = np.quantile(y_pred_exp, q1)
is_high = y_pred_exp > pred_cut

final_pred_log = np.where(
    is_high,
    pred_high,
    pred_normal
)

In [132]:
compute_metrics(y_test, final_pred_log, y_train, final_train_pred_log)

{'R2(log)': 0.9241834840057955,
 'R2': 0.8679972933264769,
 'MAPE': 11.627720125700133,
 'MdAPE': 7.616757311835332,
 'RMSE': 318828.54738971376,
 'MAE': 144910.9890880646,
 'Bias(mean residual)': 1941.7262043999651,
 'APE_95pct': 36.17230262298976,
 'APE_99pct': 64.39753995078424,
 'APE_max': 184.3296785714288,
 'Train_R2(log)': 0.9697502970602655,
 'Test_R2(log)': 0.9241834840057955,
 'R2_gap': 0.04556681305446997}